# Lab — Semantic Search

**Objective:** Build a hashing-vector search pipeline over a tiny document set.

**Book track:** `03-language-and-representation` · **Time:** 30–45 minutes · **Python:** 3.10+

Work through the cells in order. Each code cell should run top-to-bottom. Keep `main.py` in this folder aligned with your final answers—`pytest` validates that file.


## How to use this notebook

1. Open from the lab directory (`labs/02-semantic-search/`) in Jupyter, VS Code, or Codespaces.
2. Run cells sequentially; restart the kernel if you change earlier definitions.
3. Complete **Your turn** sections, then sync working code into `main.py`.
4. Run the verification cell (`pytest`) before you finish.


In [ ]:
from pathlib import Path

LAB_DIR = Path('.').resolve()
assert (LAB_DIR / 'main.py').exists(), (
    'Start Jupyter from the lab directory, e.g. labs/02-semantic-search/'
)
print('Lab directory:', LAB_DIR)


## Tasks

1. Inspect token buckets and explain why paraphrases score higher than unrelated docs.
2. Add a hard-negative document that shares tokens but wrong intent.
3. Measure recall@1 on five hand-written queries.
4. List what breaks if you change embedding dimensions.


## Step 1 — Tokenize documents

We build a tiny lexical index without external embedding APIs. Tokens are lowercased alphanumeric words.


In [ ]:
import re
from collections import Counter

DOCUMENTS = [
    'Reset a forgotten employee password in the identity portal.',
    'Investigate an unavailable application and service outage.',
    'Submit and approve an expense reimbursement invoice.',
    'Request access to a restricted analytics database.',
]


def tokens(text: str) -> list[str]:
    return re.findall(r'[a-z0-9]+', text.lower())


for doc in DOCUMENTS:
    print(tokens(doc)[:8], '...')


## Step 2 — Hashing trick embeddings

Map each token into a fixed-size vector using a stable hash bucket. This is not production quality—it teaches the retrieval pipeline shape.


In [ ]:
from hashlib import sha256


def embed(text: str, dimensions: int = 32) -> list[float]:
    counts = Counter(tokens(text))
    vector = [0.0] * dimensions
    for token, count in counts.items():
        bucket = int.from_bytes(sha256(token.encode()).digest()[:4], 'big') % dimensions
        vector[bucket] += count
    return vector


print('embedding length:', len(embed(DOCUMENTS[0])))


## Step 3 — Cosine search over the corpus


In [ ]:
from math import sqrt


def cosine(a: list[float], b: list[float]) -> float:
    dot = sum(x * y for x, y in zip(a, b))
    na, nb = sqrt(sum(x * x for x in a)), sqrt(sum(y * y for y in b))
    return dot / (na * nb) if na and nb else 0.0


def search(query: str) -> list[tuple[float, str]]:
    q = embed(query)
    return sorted(((cosine(q, embed(doc)), doc) for doc in DOCUMENTS), reverse=True)


for score, doc in search('the application is unavailable'):
    print(f'{score:.3f}  {doc}')


## Your turn

1. Explain why the outage document ranks above unrelated docs.
2. Add a **hard-negative** document that shares tokens but wrong intent.
3. Measure **recall@1** on five hand-written queries.
4. Change `dimensions`—what breaks?

Sync `embed`, `cosine`, and `search` into `main.py` when done.


In [ ]:
queries = [
    'the application is unavailable',
    'password reset portal',
    'expense reimbursement',
    # add two more queries
]

for q in queries:
    top = search(q)[0]
    print(q, '->', top[1][:50], f'({top[0]:.3f})')


## Verify

Run the test suite against `main.py` and `test_lab.py`.


In [ ]:
import subprocess
import sys

result = subprocess.run(
    [sys.executable, '-m', 'pytest', 'test_lab.py', '-q'],
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.stderr:
    print(result.stderr, file=sys.stderr)
assert result.returncode == 0, 'Tests failed—see output above'


## Reflection

- What broke first when you changed inputs?
- Which simpler baseline would you compare against in a design review?

## Extensions

- Add another case to `test_lab.py`.
- Link observations to a concept card on the AIEBOK site.
